# HYG vs JNK Macro-Filtered Pairs Strategy

Build and test a beta-hedged return-spread strategy for `HYG` vs `JNK` using ETF Terminal price data and macro features.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from db.connection import get_engine
from stores.market import PriceStore
from stores.macro import MacroFeatureStore

In [ ]:
# Strategy constants
DATA_BACKEND = "local"
APP_ENV = "uat"
PAIR = ("HYG", "JNK")
START_DATE = "2022-01-01"

BETA_WINDOW = 30
Z_WINDOW = 30
ENTRY_Z = 3.0
EXIT_Z = 0.5
STOP_Z = 3.5

HY_OAS_CHANGE_5D_BOUNDS = (-0.15, 0.15)
HY_OAS_Z60_BOUNDS = (-2.0, 2.0)
UST_10Y_CHANGE_20D_BOUNDS = (-0.40, 0.40)

FULL_SIZE_HY_OAS_Z60_BOUNDS = (-1.0, 1.0)
HALF_SIZE_HY_MINUS_IG_OAS_Z20_THRESHOLD = 1.5
HALF_SIZE_UST_10Y_CHANGE_20D_ABS_BOUNDS = (0.20, 0.40)
FULL_SIZE = 1.0
DEFAULT_SIZE = 0.75
HALF_SIZE = 0.5

TRADING_DAYS = 252
BPS_SCALE = 10_000
PLOT_PATH = Path("pairs_macro_strategy.png")

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
price_store = PriceStore(engine)
macro_feature_store = MacroFeatureStore(engine)

feature_names = [
    "HY_OAS_CHANGE_5D",
    "HY_OAS_Z60",
    "UST_10Y_CHANGE_20D",
    "SINGLE_B_MINUS_HY_OAS",
    "HY_MINUS_IG_OAS_Z20",
]

histories = price_store.get_multi_ticker_price_history(list(PAIR), start_date=START_DATE)
prices = pd.DataFrame({ticker: frame["adj_close"] for ticker, frame in histories.items()}).dropna()
macro_features = macro_feature_store.get_feature_matrix(feature_names=feature_names, start_date=START_DATE)

prices.tail()

In [ ]:
# If you already have aligned `prices` and `macro_features` DataFrames in memory,
# you can skip the loading cell above and start from here.

returns = np.log(prices).diff().rename(columns={PAIR[0]: "HYG_ret", PAIR[1]: "JNK_ret"})

rolling_cov = returns["HYG_ret"].rolling(BETA_WINDOW).cov(returns["JNK_ret"])
rolling_var = returns["JNK_ret"].rolling(BETA_WINDOW).var()
beta = (rolling_cov / rolling_var).replace([np.inf, -np.inf], np.nan).rename("beta")

spread = (returns["HYG_ret"] - beta * returns["JNK_ret"]).rename("spread")
spread_mean = spread.rolling(Z_WINDOW).mean()
spread_std = spread.rolling(Z_WINDOW).std()
zscore = ((spread - spread_mean) / spread_std.replace(0, np.nan)).rename("zscore")

single_b_roll_mean = macro_features["SINGLE_B_MINUS_HY_OAS"].rolling(60).mean()

strategy_df = pd.concat(
    [prices, returns, beta, spread, zscore, macro_features, single_b_roll_mean.rename("single_b_roll_mean_60d")],
    axis=1,
).dropna()

strategy_df[["HYG_ret", "JNK_ret", "beta", "spread", "zscore"]].tail()

In [ ]:
def regime_is_green(row: pd.Series) -> bool:
    return (
        HY_OAS_CHANGE_5D_BOUNDS[0] <= row["HY_OAS_CHANGE_5D"] <= HY_OAS_CHANGE_5D_BOUNDS[1]
        and HY_OAS_Z60_BOUNDS[0] <= row["HY_OAS_Z60"] <= HY_OAS_Z60_BOUNDS[1]
        and UST_10Y_CHANGE_20D_BOUNDS[0] <= row["UST_10Y_CHANGE_20D"] <= UST_10Y_CHANGE_20D_BOUNDS[1]
    )


def position_size(row: pd.Series) -> float:
    full_size_ok = (
        FULL_SIZE_HY_OAS_Z60_BOUNDS[0] <= row["HY_OAS_Z60"] <= FULL_SIZE_HY_OAS_Z60_BOUNDS[1]
        and row["SINGLE_B_MINUS_HY_OAS"] < row["single_b_roll_mean_60d"]
    )
    half_size_ok = (
        row["HY_MINUS_IG_OAS_Z20"] > HALF_SIZE_HY_MINUS_IG_OAS_Z20_THRESHOLD
        or HALF_SIZE_UST_10Y_CHANGE_20D_ABS_BOUNDS[0]
        <= abs(row["UST_10Y_CHANGE_20D"])
        <= HALF_SIZE_UST_10Y_CHANGE_20D_ABS_BOUNDS[1]
    )
    if full_size_ok:
        return FULL_SIZE
    if half_size_ok:
        return HALF_SIZE
    return DEFAULT_SIZE


state = []
signal = 0
size = 0.0

for date, row in strategy_df.iterrows():
    regime_green = regime_is_green(row)
    z = row["zscore"]
    next_signal = signal
    next_size = size
    trade_event = "hold"

    if signal == 0:
        next_size = 0.0
        if regime_green and z < -ENTRY_Z:
            next_signal = 1
            next_size = position_size(row)
            trade_event = "enter_long"
        elif regime_green and z > ENTRY_Z:
            next_signal = -1
            next_size = position_size(row)
            trade_event = "enter_short"
    elif signal == 1:
        if z >= -EXIT_Z:
            next_signal = 0
            next_size = 0.0
            trade_event = "exit_long"
        elif z <= -STOP_Z:
            next_signal = 0
            next_size = 0.0
            trade_event = "stop_long"
    elif signal == -1:
        if z <= EXIT_Z:
            next_signal = 0
            next_size = 0.0
            trade_event = "exit_short"
        elif z >= STOP_Z:
            next_signal = 0
            next_size = 0.0
            trade_event = "stop_short"

    state.append(
        {
            "date": date,
            "regime_green": regime_green,
            "signal": next_signal,
            "position_size": next_size,
            "trade_event": trade_event,
        }
    )
    signal = next_signal
    size = next_size

state_df = pd.DataFrame(state).set_index("date")
strategy_df = strategy_df.join(state_df)

hyg_mean = strategy_df[PAIR[0]].mean()
jnk_mean = strategy_df[PAIR[1]].mean()
round_trip_cost = ((0.01 / hyg_mean) + (0.01 / jnk_mean)) * 2

strategy_df["gross_pnl"] = strategy_df["position_size"] * strategy_df["signal"] * strategy_df["spread"]
strategy_df["position_change"] = (
    strategy_df[["signal", "position_size"]]
    .fillna(0.0)
    .ne(strategy_df[["signal", "position_size"]].fillna(0.0).shift())
    .any(axis=1)
    .astype(int)
)
strategy_df.loc[strategy_df.index[0], "position_change"] = int(
    strategy_df.iloc[0][["signal", "position_size"]].ne(0).any()
)
strategy_df["transaction_cost"] = strategy_df["position_change"] * round_trip_cost
strategy_df["net_pnl"] = strategy_df["gross_pnl"] - strategy_df["transaction_cost"]
strategy_df[["signal", "position_size", "gross_pnl", "net_pnl"]].tail()

In [ ]:
def strategy_metrics(net_series: pd.Series, gross_series: pd.Series, signal_series: pd.Series, trade_events: pd.Series) -> pd.DataFrame:
    net_series = net_series.fillna(0.0)
    gross_series = gross_series.fillna(0.0)
    cumulative = net_series.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    annual_return_bps = net_series.mean() * TRADING_DAYS * BPS_SCALE
    annual_vol_bps = net_series.std(ddof=0) * np.sqrt(TRADING_DAYS) * BPS_SCALE
    sharpe = np.nan if annual_vol_bps == 0 else annual_return_bps / annual_vol_bps
    max_drawdown_bps = drawdown.min() * BPS_SCALE

    entries = trade_events.isin(["enter_long", "enter_short"])
    exits = trade_events.isin(["exit_long", "exit_short", "stop_long", "stop_short"])
    trades = int(entries.sum())

    active_trade_pnls = []
    current_trade = 0.0
    in_trade = False
    for pnl, event in zip(net_series, trade_events):
        if event in {"enter_long", "enter_short"}:
            current_trade = pnl
            in_trade = True
            continue
        if in_trade:
            current_trade += pnl
            if event in {"exit_long", "exit_short", "stop_long", "stop_short"}:
                active_trade_pnls.append(current_trade)
                current_trade = 0.0
                in_trade = False

    hit_rate = np.nan
    if active_trade_pnls:
        hit_rate = 100 * (np.array(active_trade_pnls) > 0).mean()

    summary = pd.DataFrame(
        {
            "metric": [
                "Annualized Return (bps)",
                "Annualized Vol (bps)",
                "Sharpe Ratio",
                "Max Drawdown (bps)",
                "Number of Trades",
                "Time in Market (%)",
                "Hit Rate (%)",
                "Round-Trip Cost (return units)",
            ],
            "value": [
                annual_return_bps,
                annual_vol_bps,
                sharpe,
                max_drawdown_bps,
                trades,
                100 * (signal_series != 0).mean(),
                hit_rate,
                round_trip_cost,
            ],
        }
    )
    return summary


summary_table = strategy_metrics(
    net_series=strategy_df["net_pnl"],
    gross_series=strategy_df["gross_pnl"],
    signal_series=strategy_df["signal"],
    trade_events=strategy_df["trade_event"],
)

summary_table.style.format({"value": "{:.2f}"})

In [ ]:
cum_net_bps = strategy_df["net_pnl"].cumsum() * BPS_SCALE
gray_mask = ~strategy_df["regime_green"]

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True, constrained_layout=True)

axes[0].plot(strategy_df.index, strategy_df["zscore"], color="navy", lw=1.4, label="Spread z-score")
axes[0].axhline(ENTRY_Z, color="firebrick", ls="--", lw=1.0, label="Entry band")
axes[0].axhline(-ENTRY_Z, color="firebrick", ls="--", lw=1.0)
axes[0].axhline(EXIT_Z, color="gray", ls=":", lw=1.0, label="Exit band")
axes[0].axhline(-EXIT_Z, color="gray", ls=":", lw=1.0)
axes[0].axhline(STOP_Z, color="darkorange", ls="--", lw=1.0, label="Stop band")
axes[0].axhline(-STOP_Z, color="darkorange", ls="--", lw=1.0)
axes[0].fill_between(strategy_df.index, -6, 6, where=gray_mask, color="lightgray", alpha=0.25, label="Regime blocked")
axes[0].set_title("HYG vs JNK Spread Z-Score")
axes[0].set_ylabel("z-score")
axes[0].set_ylim(-6, 6)
axes[0].legend(loc="upper left", ncol=2)

axes[1].step(strategy_df.index, strategy_df["signal"] * strategy_df["position_size"], where="post", color="purple", lw=1.4)
axes[1].fill_between(strategy_df.index, 0, strategy_df["signal"] * strategy_df["position_size"], step="post", alpha=0.20, color="mediumpurple")
axes[1].set_title("Position Size")
axes[1].set_ylabel("signed size")

axes[2].plot(strategy_df.index, cum_net_bps, color="darkgreen", lw=1.6)
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_title("Cumulative Net P&L")
axes[2].set_ylabel("bps")

for ax in axes:
    ax.grid(alpha=0.25)

plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()

print(f"saved plot to {PLOT_PATH.resolve()}")